In [1]:
import pandas as pd

data = pd.read_csv(r'D:\My_Space\College\6th_SEM\Deep_Learning_Project\IDS\Data\processed\cicids2017_merged.csv')

In [2]:
def group_labels(label):
    
    if label == 'BENIGN':
        return 'BENIGN'
    
    elif 'DoS' in label and 'DDoS' not in label:
        return 'DoS'
    
    elif 'DDoS' in label:
        return 'DDoS'
    
    elif 'PortScan' in label:
        return 'PortScan'
    
    elif 'Patator' in label:
        return 'BruteForce'
    
    elif 'Web Attack' in label:
        return 'WebAttack'
    
    elif 'Bot' in label:
        return 'Bot'
    
    elif 'Infiltration' in label:
        return 'Infiltration'
    
    elif 'Heartbleed' in label:
        return 'Heartbleed'
    
    else:
        return 'Other'

data['Attack_Category'] = data['Label'].apply(group_labels)

data = data[~data['Attack_Category'].isin(['Infiltration', 'Heartbleed'])]

print(data['Attack_Category'].value_counts())

Attack_Category
BENIGN        2095057
DoS            193745
DDoS           128014
PortScan        90694
BruteForce       9150
WebAttack        2143
Bot              1948
Name: count, dtype: int64


In [3]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
data['Encoded_Label'] = le.fit_transform(data['Attack_Category'])

In [4]:
X = data.drop(['Label', 'Attack_Category', 'Encoded_Label'], axis=1)
y = data['Encoded_Label']

<h2><center>FEATURE REDUCTION STARTS HERE<center></h2>

</h3>STEP 1 – Remove Zero Variance Features</h3>

In [5]:
from sklearn.feature_selection import VarianceThreshold

selector = VarianceThreshold(threshold=0.0)
X_var = selector.fit_transform(X)

selected_columns = X.columns[selector.get_support()]

X = pd.DataFrame(X_var, columns=selected_columns)

print("After removing zero variance:", X.shape)

After removing zero variance: (2520751, 70)


<h3>STEP 2 – Remove Highly Correlated Features</h3>

In [6]:
import numpy as np
corr_matrix = X.corr().abs()

upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

to_drop = [column for column in upper_triangle.columns if any(upper_triangle[column] > 0.95)]

print("Highly correlated features:", len(to_drop))

X.drop(columns=to_drop, inplace=True)

print("After correlation filtering:", X.shape)

Highly correlated features: 23
After correlation filtering: (2520751, 47)


<h3>STEP 3 – Random Forest Feature Importance</h3>

In [7]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=50,
    n_jobs=-1,
    random_state=42
)

rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)

importances.head(20)

Packet Length Variance         0.117682
Bwd Packet Length Max          0.087176
Max Packet Length              0.072806
Fwd Packet Length Max          0.059297
Total Length of Fwd Packets    0.054717
Flow IAT Max                   0.052441
Packet Length Mean             0.047787
act_data_pkt_fwd               0.035217
Destination Port               0.032213
Init_Win_bytes_backward        0.030455
Fwd IAT Std                    0.028856
Total Fwd Packets              0.028514
Fwd Packet Length Mean         0.027959
Flow IAT Std                   0.026003
PSH Flag Count                 0.024402
Fwd Header Length              0.023184
Flow Packets/s                 0.022730
Bwd Header Length              0.022617
Init_Win_bytes_forward         0.020979
Bwd Packets/s                  0.017332
dtype: float64

<h3>STEP 4 – Select Top 35 Features</h3>

In [8]:
top_features = importances.head(35).index

X_reduced = X[top_features]

print("Final selected feature shape:", X_reduced.shape)

Final selected feature shape: (2520751, 35)


<h3>STEP 5 – Save Reduced Dataset</h3>

In [9]:
import os

X_reduced = X[top_features].copy()
# Add encoded labels
X_reduced['Encoded_Label'] = y.values

# Define save path (aligned with your project structure)
save_path = r"D:\My_Space\College\6th_SEM\Deep_Learning_Project\IDS\Data\processed_04"
os.makedirs(save_path, exist_ok=True)

# Save cleaned dataset
X_reduced.to_csv(os.path.join(save_path, "cicids2017_selected_top_features.csv"), index=False)

print("\nDataset saved successfully at:", os.path.join(save_path, "cicids2017_selected_top_features.csv"))


Dataset saved successfully at: D:\My_Space\College\6th_SEM\Deep_Learning_Project\IDS\Data\processed_04\cicids2017_selected_top_features.csv
